<a href="https://colab.research.google.com/github/chenweioh/GCP-Inspector-Toolkit/blob/main/PK_Pairwise_Comparison_Analysis_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# User Guide for AUC, Cmax, and Pairwise Comparison Tool

## Overview
This Colab-based tool is designed for comprehensive pharmacokinetic (PK) data analysis. It not only calculates Area Under Curve (AUC) and Maximum Concentration (Cmax) from time-concentration data but also produces pairwise comparison graphs. These pairwise graphs are invaluable for comparing concentration-time curves of different subjects. They aid in the verification of data integrity by helping to identify if the same sample has been re-analyzed but presented as a new subject.

## Prerequisites
- Google Colab environment (no need to install any software; it runs in the browser).
- Basic familiarity with Google Colab or Jupyter Notebooks.

## Excel File Requirements
The Excel file must contain the following columns (case-sensitive):
- `subject`: Identifies the subject number.
- `period`: Specifies the period of measurement for each subject.
- `time`: Records the time of each measurement in hours.
- `concentration`: Lists the concentration levels measured.

**Note:** Missing concentration values should be left blank. The tool will automatically replace them with zeros.

## Steps to Use the Tool

### Step 1: Upload Your Excel File
Click on the upload button in the Colab interface to upload your Excel .xlsx file.

### Step 2: Run the Analysis
Once the file is uploaded, click the "Run" button to perform the analysis.

### Step 3: Review the Output
After clicking "Run," the tool will loop through each subject and each period, calculating the AUC and Cmax. For each subject and period, the tool will also generate a plot displaying the concentration-time curve.

#### Pairwise Comparisons
This is a highlight feature of the tool. After analyzing individual subjects, it will generate pairwise comparison graphs between the concentration-time curves of all possible subject pairs. This feature aids in visually inspecting similarities or differences between subjects.

### Step 4: Debug (If Necessary)
If you see any anomalies or unexpected behavior, check the Excel file for incorrect or missing values and re-upload.

## Troubleshooting
- Ensure the Excel file is properly formatted according to the requirements above.
- Make sure the uploaded file is not open in any other program while uploading to prevent any read/write conflicts.

## Final Notes
Dive deep into your data and make the most of the pairwise comparison feature to derive insights. Happy data analyzing!


In [ ]:
import pandas as pd
from itertools import combinations
import matplotlib.pyplot as plt
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display

# Function for pairwise comparison
def pairwise_comparison(df, max_subject_number):
    for idx, (subj1, subj2) in enumerate(combinations(range(1, max_subject_number + 1), 2)):
        print(f"Comparing subject {subj1} with {subj2}")

        df_subj1 = df[(df['subject'] == subj1)]
        df_subj2 = df[(df['subject'] == subj2)]

        Time_hrs1 = df_subj1['time'].tolist()
        Conc1 = df_subj1['concentration'].tolist()
        Time_hrs2 = df_subj2['time'].tolist()
        Conc2 = df_subj2['concentration'].tolist()

        plt.figure(figsize=(10, 6))

        # Plotting subject 1
        plt.plot(Time_hrs1, Conc1, label=f'Subject {subj1}')

        # Plotting subject 2
        plt.plot(Time_hrs2, Conc2, label=f'Subject {subj2}')

        plt.xlabel('Time (hrs)')
        plt.ylabel('Concentration')
        plt.title(f'Pairwise Comparison: {subj1} vs {subj2}')
        plt.legend()
        plt.grid(True)

        plt.show()

# Upload widget
uploader = widgets.FileUpload(
    accept='.xlsx',
    multiple=False
)

# Run button
run_button = widgets.Button(description="Run")

# Function to run when file is uploaded
def on_run_button_clicked(b):
    if uploader.value:
        uploaded_file = uploader.value[list(uploader.value.keys())[0]]['content']
        bytes_data = BytesIO(uploaded_file)
        df = pd.read_excel(bytes_data)
        # Fill NaN values with 0
        df['concentration'] = df['concentration'].fillna(0)
        # Get maximum subject number
        max_subject_number = df['subject'].max()

        # Loop through each subject and period to calculate AUC and Cmax
        for subj in range(1, max_subject_number + 1):
            df_subj = df[df['subject'] == subj]
            for period in df_subj['period'].unique():
                df_period = df_subj[df_subj['period'] == period]

                # Time and Concentration
                Time_hrs = df_period['time'].tolist()
                Conc = df_period['concentration'].tolist()

                # Calculate AUC
                AUC_segments = []
                for i in range(1, len(Time_hrs)):
                    AUC_segment = 0.5 * (Conc[i] + Conc[i-1]) * (Time_hrs[i] - Time_hrs[i-1])
                    AUC_segments.append(AUC_segment)
                total_AUC = sum(AUC_segments)

                # Calculate Cmax and Tmax
                Cmax = max(Conc)
                Tmax = Time_hrs[Conc.index(Cmax)]

                print(f"Subject {subj}, Period {period}: Total AUC: {total_AUC}, Cmax: {Cmax}, Tmax: {Tmax}")

                # Plotting
                plt.figure(figsize=(10, 6))
                plt.scatter(Time_hrs, Conc, c='red')
                plt.plot(Time_hrs, Conc, label=f'Subject {subj}, Period {period}')
                plt.xlabel('Time (hrs)')
                plt.ylabel('Concentration')
                plt.title('Concentration vs Time')
                plt.legend()
                plt.grid(True)
                plt.show()

        # Call the pairwise comparison function
        pairwise_comparison(df, max_subject_number)

    else:
        print("Please upload an Excel file first.")

run_button.on_click(on_run_button_clicked)

# Display widgets
display(uploader, run_button)


FileUpload(value={}, accept='.xlsx', description='Upload')

Button(description='Run', style=ButtonStyle())